# URBANopt CLI Docker Smoke Test

This notebook builds a Docker image with URBANopt CLI 1.2.0 and runs quick test commands.
It also demonstrates a Python helper that mounts local `esbe_2026` into the container.

In [5]:
from pathlib import Path
import subprocess

REPO_ROOT = Path.cwd()
DOCKERFILE = REPO_ROOT / 'docker' / 'urbanopt-cli' / 'Dockerfile'
IMAGE_TAG = 'urbanopt-cli:1.2-ubuntu22'

print('Repo root:', REPO_ROOT)
print('Dockerfile:', DOCKERFILE)
print('Image tag:', IMAGE_TAG)
subprocess.run(['docker', '--version'], check=False)

Repo root: /Users/nlong/working/urban-analysis/esbe
Dockerfile: /Users/nlong/working/urban-analysis/esbe/docker/urbanopt-cli/Dockerfile
Image tag: urbanopt-cli:1.2-ubuntu22
Docker version 29.1.5, build 0e6fee6


CompletedProcess(args=['docker', '--version'], returncode=0)

In [6]:
# Build the Docker image
build_cmd = [
    'docker', 'build',
    '-t', IMAGE_TAG,
    '-f', str(DOCKERFILE),
    str(REPO_ROOT),
]
print('Building Docker image with command:', ' '.join(build_cmd))
subprocess.run(build_cmd, check=True)

Building Docker image with command: docker build -t urbanopt-cli:1.2-ubuntu22 -f /Users/nlong/working/urban-analysis/esbe/docker/urbanopt-cli/Dockerfile /Users/nlong/working/urban-analysis/esbe


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 1.19kB done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/ubuntu:22.04
#2 ...

#3 [auth] library/ubuntu:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 [internal] load metadata for docker.io/library/ubuntu:22.04
#2 DONE 0.9s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/4] FROM docker.io/library/ubuntu:22.04@sha256:962f6cadeae0ea6284001009daa4cc9a8c37e75d1f5191cf0eb83fe565b63dd7
#5 DONE 0.0s

#6 [2/4] RUN apt-get update && apt-get install -y --no-install-recommends     curl     ca-certificates     bash     && rm -rf /var/lib/apt/lists/*
#6 CACHED

#7 [3/4] RUN set -eux;     ARCH="$(dpkg --print-architecture)";     if [ "$ARCH" = "amd64" ]; then UO_ARCH="x86_64";     elif [ "$ARCH" = "arm64" ]; then UO_ARCH="arm64";     else echo "Unsupported architecture: $ARCH" >&2; exit 1; fi;    

CompletedProcess(args=['docker', 'build', '-t', 'urbanopt-cli:1.2-ubuntu22', '-f', '/Users/nlong/working/urban-analysis/esbe/docker/urbanopt-cli/Dockerfile', '/Users/nlong/working/urban-analysis/esbe'], returncode=0)

In [8]:
# Run basic URBANopt CLI smoke tests
subprocess.run(['docker', 'run', '--rm', IMAGE_TAG, 'uo', '--version'], check=True)
subprocess.run(['docker', 'run', '--rm', IMAGE_TAG, 'uo', '--help'], check=True)

1.2.0

URBANopt CLI version: 1.2.0

Usage:
  uo [options] [<command> [suboptions]]
 
Options:
  -v, --version    Print version and exit
  -h, --help       Show this help message

Commands:
  create         Make new things - project directory or files
  install_python Install python and other dependencies to run OpenDSS, DISCO,
GMT analysis
  update         Update files in an existing URBANopt project
  run            Use files in your directory to simulate district energy use
  process        Post-process URBANopt simulations for additional insights
  visualize      Visualize and compare results for features and scenarios
  validate       Validate results with custom rules
  opendss        Run OpenDSS simulation
  disco          Run DISCO analysis
  rnm            Run RNM simulation
  delete         Delete simulations for a specified scenario
  des_params     Make a DES system parameters config file
  des_create     Create a Modelica model
  des_run        Run a Modelica DES model
  gh

CompletedProcess(args=['docker', 'run', '--rm', 'urbanopt-cli:1.2-ubuntu22', 'uo', '--help'], returncode=0)

In [9]:
# Use the helper script to run against local esbe_2026 files
from tools.urbanopt_docker import UrbanoptDockerRunner

runner = UrbanoptDockerRunner(image_tag=IMAGE_TAG, mount_subdir='esbe_2026')
result = runner.run_uo(['--help'])
print('Exit code:', result.returncode)
print(result.stdout[:1200])
if result.stderr:
    print('stderr:', result.stderr[:1200])

Exit code: 0

URBANopt CLI version: 1.2.0

Usage:
  uo [options] [<command> [suboptions]]
 
Options:
  -v, --version    Print version and exit
  -h, --help       Show this help message

Commands:
  create         Make new things - project directory or files
  install_python Install python and other dependencies to run OpenDSS, DISCO,
GMT analysis
  update         Update files in an existing URBANopt project
  run            Use files in your directory to simulate district energy use
  process        Post-process URBANopt simulations for additional insights
  visualize      Visualize and compare results for features and scenarios
  validate       Validate results with custom rules
  opendss        Run OpenDSS simulation
  disco          Run DISCO analysis
  rnm            Run RNM simulation
  delete         Delete simulations for a specified scenario
  des_params     Make a DES system parameters config file
  des_create     Create a Modelica model
  des_run        Run a Modelica DES mod

In [10]:
# Example shell command from inside mounted esbe_2026
ls_result = runner.run_shell('pwd && ls -la | head -n 30')
print('Exit code:', ls_result.returncode)
print(ls_result.stdout)
if ls_result.stderr:
    print('stderr:', ls_result.stderr)

Exit code: 0
/workspace/esbe_2026
total 4
drwxr-xr-x 4 root root  128 Apr 22 16:37 .
drwxr-xr-x 1 root root 4096 Apr 24 13:09 ..
drwxr-xr-x 3 root root   96 Apr 22 16:36 activity_0
drwxr-xr-x 6 root root  192 Apr 22 16:37 activity_1

